# gerador simples

Testar:
- sleep baixo e elevado no gerador
- timeout
- done 

In [0]:
import time

def gen():
  for i in range(5):
    time.sleep(60)
    yield i

# query_endpoint

In [0]:
from mlflow.deployments import get_deploy_client
from databricks.sdk import WorkspaceClient
import json
import uuid

import logging

In [0]:
def _convert_to_responses_format(messages):
    """Convert chat messages to ResponsesAgent API format."""
    input_messages = []
    for msg in messages:
        if msg["role"] == "user":
            input_messages.append({"role": "user", "content": msg["content"]})
        elif msg["role"] == "assistant":
            # Handle assistant messages with tool calls
            if msg.get("tool_calls"):
                # Add function calls
                for tool_call in msg["tool_calls"]:
                    input_messages.append({
                        "type": "function_call",
                        "id": tool_call["id"],
                        "call_id": tool_call["id"],
                        "name": tool_call["function"]["name"],
                        "arguments": tool_call["function"]["arguments"]
                    })
                # Add assistant message if it has content
                if msg.get("content"):
                    input_messages.append({
                        "type": "message",
                        "id": msg.get("id", str(uuid.uuid4())),
                        "content": [{"type": "output_text", "text": msg["content"]}],
                        "role": "assistant"
                    })
            else:
                # Regular assistant message
                input_messages.append({
                    "type": "message",
                    "id": msg.get("id", str(uuid.uuid4())),
                    "content": [{"type": "output_text", "text": msg["content"]}],
                    "role": "assistant"
                })
        elif msg["role"] == "tool":
            input_messages.append({
                "type": "function_call_output",
                "call_id": msg.get("tool_call_id"),
                "output": msg["content"]
            })
    return input_messages


In [0]:
def _query_responses_endpoint_stream(endpoint_name: str, messages: list[dict[str, str]], return_traces: bool):
    """Stream responses from agent/v1/responses endpoints using MLflow deployments client."""
    client = get_deploy_client("databricks")
    
    input_messages = _convert_to_responses_format(messages)
    
    # Prepare input payload for ResponsesAgent
    inputs = {
        "input": input_messages,
        "context": {},
        "stream": True
    }
    if return_traces:
        inputs["databricks_options"] = {"return_trace": True}

    for event_data in client.predict_stream(endpoint=endpoint_name, inputs=inputs):
        # Just yield the raw event data, let app.py handle the parsing
        yield event_data

In [0]:
results = _query_responses_endpoint_stream(
  'agents_lucas_catalog-default-langgraph-mcp-responses-agent', 
  [{'role': 'user', 'content': 'qual o total de transações?'}], 
  return_traces=False
)

# threading

In [0]:
from threading import Thread

TIMEOUT_SECONDS = 2

class NextThread(Thread):

  def __init__(self, group=None, target=None, name=None, args=(), kwargs={}, arg=None):
    Thread.__init__(self, group=group, target=target, name=name, args=args, kwargs=kwargs)
    self.target = target
    self.arg = arg
    self.result = None
    self.exception = None

  def run(self):
    try:
      self.result = self.target(self.arg)
    except StopIteration as e:
      self.exception = e

  def join(self, timeout=None):
    Thread.join(self, timeout)
    if self.exception:
      raise self.exception
    if self.is_alive():
      raise TimeoutError

def process_response(results):
  last_is_tool = False
  while True:
    try:
      thread = NextThread(target=next, arg=results)
      thread.start()
      thread.join(timeout=TIMEOUT_SECONDS)
      result = thread.result
      if result["type"] == "response.output_text.delta":
          last_is_tool = False
          yield result["delta"]
      elif (result["type"] == "response.output_item.done") and (result["item"]["type"] == "function_call") and (last_is_tool == False):
          last_is_tool = True
          yield "\n\nConsultando informações...\n\n"
    except StopIteration:
      break
    except TimeoutError:
      yield "\n\nA consulta demorou mais que o tempo máximo! Tente novamente em breve..."
      break

for i in process_response(results):
  print(i)

# concurrent.futures

- Não interrompe a thread

In [0]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

TIMEOUT_SECONDS = 3

with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(gen)
    try:
        future.result(timeout=TIMEOUT_SECONDS)
    except FuturesTimeoutError:
        raise TimeoutError(f"Function did not finish within {TIMEOUT_SECONDS} seconds.")

# multiprocessing

- não funciona com gerador

In [0]:
import multiprocessing

TIMEOUT_SECONDS = 30

process = multiprocessing.Process(target=gen)
process.start()
process.join(timeout=TIMEOUT_SECONDS)

if process.is_alive():
    print(f"TIMEOUT...")
    process.terminate()
    process.join()  # Wait for termination
else:
    print("Process finished within the timeout.")